# 캡스톤. 나만의 도메인 SLM

데이터 수집 → BPE → 학습 → 평가 → 양자화 → GGUF → **HuggingFace Hub 업로드** → 데모까지 풀 사이클.
본인이 만든 모델이 다음 사람의 "기성 sLLM" 이 되는 경험.

In [ ]:
# !pip install -q huggingface_hub transformers gradio

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

# 캡스톤 체크리스트
checklist = [
    "[1] 도메인 결정 + 데이터 수집/합성 (Ch 5, 7)",
    "[2] PII 마스킹 + de-dup + 라이선스 정리 (Ch 7, 29)",
    "[3] BPE 토크나이저 훈련 (Ch 6)",
    "[4] 모델 config 결정 10M~30M (Ch 4, 11)",
    "[5] 학습 mixed precision, grad accum (Ch 12~15)",
    "[6] 평가 perplexity + probe + 회귀 (Ch 16~18, 30)",
    "[7] int4 양자화 + GGUF 변환 (Ch 19, 20)",
    "[8] HuggingFace Hub 업로드 (이 캡스톤)",
    "[9] Spaces 데모 — Gradio (선택)",
    "[10] 회고 — 다시 한다면 무엇을 바꿀 것인가",
]
for step in checklist:
    print(f"  {step}")

## 최소 예제 — HuggingFace Hub 업로드

### 사전 준비

```bash
pip install huggingface_hub
huggingface-cli login   # 토큰 입력 (Settings → Access Tokens)
```

In [ ]:
# HuggingFace Hub 로그인 확인
# from huggingface_hub import whoami
# try:
#     info = whoami()
#     print(f"로그인됨: {info['name']}")
# except Exception as e:
#     print(f"로그인 필요: {e}")
#     print("  터미널에서: huggingface-cli login")

print("huggingface-cli login 으로 로그인 후 아래 코드를 실행하세요.")

In [ ]:
# 모델 + 토크나이저 push
# from huggingface_hub import HfApi, create_repo
# from transformers import AutoTokenizer
#
# repo_id = "desty/tiny-tale-ko-10m"                    # (1) {username}/{model-name}
# create_repo(repo_id, repo_type="model", exist_ok=True)
#
# # 모델 가중치 (PyTorch state_dict 또는 safetensors 권장)
# api = HfApi()
# api.upload_folder(
#     folder_path="checkpoints/final",                  # (2) config.json, model.safetensors, tokenizer.json 등
#     repo_id=repo_id,
#     repo_type="model",
# )
#
# # (선택) GGUF 변환본도 같이 올리기                    (3)
# api.upload_file(
#     path_or_fileobj="dist/tiny-tale-ko-10m-q4.gguf",
#     path_in_repo="tiny-tale-ko-10m-q4.gguf",
#     repo_id=repo_id,
# )

# 업로드 전 체크리스트
upload_checklist = [
    "학습 데이터에 PII 마스킹 완료 (Ch 29)",
    "학습 데이터 라이선스 정리 + 모델 라이선스 결정",
    "모델 카드 7항목 + 한계 섹션 + 사용법 코드",
    "tokenizer.json 포함 확인",
    "(선택) GGUF int4 + fp16 둘 다",
    "(선택) safetensors 형식 (.bin 보다 안전)",
    "회귀 평가 통과 (Ch 30)",
    "private 으로 먼저 올려 from_pretrained 동작 확인",
    "확인 후 public 전환",
]
print("업로드 전 게이트:")
for item in upload_checklist:
    print(f"  [ ] {item}")

## 최소 예제 — Gradio 데모 (HF Spaces)

HuggingFace Spaces 에서 `app.py` 로 Gradio 5줄 데모 실행.

In [ ]:
# HF Spaces 의 app.py (5줄 데모)
# import gradio as gr
# from transformers import AutoModelForCausalLM, AutoTokenizer
#
# tok = AutoTokenizer.from_pretrained("desty/tiny-tale-ko-10m")
# m = AutoModelForCausalLM.from_pretrained("desty/tiny-tale-ko-10m")
#
# def gen(prompt):
#     ids = tok(prompt, return_tensors="pt").input_ids
#     out = m.generate(ids, max_new_tokens=120, do_sample=True, top_p=0.9, temperature=0.8)
#     return tok.decode(out[0], skip_special_tokens=True)
#
# gr.Interface(fn=gen, inputs="text", outputs="text",
#              title="Tiny Tale KO 10M").launch()

# Colab 에서 로컬 테스트용 (모델 로드 후)
# gr.Interface(fn=gen, inputs="text", outputs="text",
#              title="Tiny Tale KO 10M").launch(share=True)  # share=True 로 임시 URL 생성

print("HF Spaces 에 app.py 를 올리면 무료 공개 데모가 생성됩니다.")
print("https://huggingface.co/spaces/{username}/{space-name}")

## 실전 — 모델 카드 (README.md) 생성

HF Hub 의 첫 페이지. Ch 22 의 7항목을 본인이 채운다.

In [ ]:
MODEL_CARD_TEMPLATE = """---
license: apache-2.0
language:
  - ko
tags:
  - text-generation
  - small-language-model
  - tinystories
  - korean
datasets:
  - {username}/tinystories-ko-synthetic
base_model: null
---

# Tiny Tale KO 10M

A 10M-parameter Korean fairy-tale generator, trained from scratch as the
capstone of [Tiny LLM from Scratch](https://desty.github.io/study-tiny-llm/).

## 모델 7항목

| 항목 | 값 |
|---|---|
| 전체 / 활성 파라미터 | 10M / 10M (dense) |
| 학습 토큰 | 200M (Chinchilla 20×) |
| 학습 데이터 | TinyStories-KO 합성 (50K 동화) |
| 컨텍스트 길이 | 512 |
| 라이선스 | Apache 2.0 |
| 토크나이저 | BPE 8K vocab (한글 자모 분리) |
| 양자화 | fp16, int4 GGUF 제공 |

## 사용법

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained("{username}/tiny-tale-ko-10m")
m = AutoModelForCausalLM.from_pretrained("{username}/tiny-tale-ko-10m")
```

llama.cpp:

```bash
llama-cli -m tiny-tale-ko-10m-q4.gguf -p "옛날 옛적에"
```

## 한계

- 도메인이 좁음 — 동화 외 입력에는 깨짐
- 컨텍스트 512 — RAG 부적합
- 한국어만 학습 — 영어 깨짐
"""

# 본인 정보로 채우기
username = "your-hf-username"  # 본인 HF 사용자명으로 변경
model_card = MODEL_CARD_TEMPLATE.format(username=username)
print(model_card[:500] + "...\n(전체 내용 생략)")

## 실전 — 풀 사이클 체크포인트 자동화

캡스톤 10단계 자동 검증.

In [ ]:
from pathlib import Path
import json

def check_capstone_gates(checkpoint_dir, gguf_path=None, meta_path=None):
    """업로드 전 마지막 게이트 자동 검사."""
    gates = {}
    ckpt = Path(checkpoint_dir)

    # 필수 파일 존재 확인
    gates["config.json"]            = (ckpt / "config.json").exists()
    gates["tokenizer.json"]         = (ckpt / "tokenizer.json").exists()
    gates["tokenizer_config.json"]  = (ckpt / "tokenizer_config.json").exists()
    gates["model 가중치"]            = any(ckpt.glob("*.safetensors")) or any(ckpt.glob("*.bin"))

    # 선택 항목
    if gguf_path:
        gates["GGUF 파일"] = Path(gguf_path).exists()

    if meta_path:
        meta_exists = Path(meta_path).exists()
        gates["메타파일 (PII/IAA)"] = meta_exists
        if meta_exists:
            meta = json.loads(Path(meta_path).read_text())
            gates["IAA κ ≥ 0.6"] = meta.get("iaa_kappa", 0) >= 0.6

    # 결과 출력
    all_pass = True
    for gate, ok in gates.items():
        status = "OK" if ok else "FAIL"
        if not ok:
            all_pass = False
        print(f"  [{status}] {gate}")

    print()
    if all_pass:
        print("모든 게이트 통과 — 업로드 준비 완료")
    else:
        print("일부 게이트 실패 — 위 항목 확인 필요")
    return all_pass

# 테스트 (실제 경로로 변경)
# check_capstone_gates(
#     checkpoint_dir="checkpoints/final",
#     gguf_path="dist/tiny-tale-ko-10m-q4.gguf",
#     meta_path="data/meta.json",
# )

# 데모: 임시 디렉토리로 테스트
import tempfile
with tempfile.TemporaryDirectory() as tmpdir:
    # 필수 파일 생성
    Path(f"{tmpdir}/config.json").write_text('{"model_type":"gpt2"}')
    Path(f"{tmpdir}/tokenizer.json").write_text('{}')
    Path(f"{tmpdir}/tokenizer_config.json").write_text('{}')
    Path(f"{tmpdir}/model.safetensors").write_bytes(b"dummy")

    print("체크포인트 게이트 검사:")
    check_capstone_gates(tmpdir)

## 회고 (마지막 한 페이지)

업로드 후 아래 질문에 답하며 한 페이지를 작성한다. **다시 한다면 무엇을 바꿀 것인가.**

- **데이터** — 합성 비중을 더 늘릴까? 사람 검수 비중은?
- **모델 크기** — 10M 이 적정이었나, 30M 이었어야 하나?
- **학습 시간** — over-training 100× 까지 갔어야 하나?
- **평가** — 어느 probe 가 가장 유용했나?
- **양자화** — int4 손실은 도메인에서 얼마였나?
- **카드** — 어느 항목이 추가됐어야 하나?

이 회고가 **다음 모델 만들 때의 시작점**이다.

In [ ]:
# 회고 작성 (자유 형식)
retrospective = {
    "데이터": "",      # 예: "합성 비중 70% → 90% 로 늘렸어야"
    "모델 크기": "",   # 예: "10M 으로 충분했음. 30M 은 Colab 시간 초과"
    "학습 시간": "",
    "평가": "",
    "양자화": "",
    "카드": "",
    "다음에 바꿀 것": "",
}

print("회고 템플릿 (아래 빈칸을 채우세요):")
for k, v in retrospective.items():
    print(f"  {k}: {v if v else '(작성 필요)'}")

## 연습문제 (캡스톤 전 과정)

캡스톤은 연습이 아니라 **실제 프로젝트**다. 아래 순서대로 완주하자.

1. **도메인 선택**: 한국 동화 생성기 / 레시피 도우미 / 커밋 메시지 / 도메인 NER 중 하나.
2. **데이터 준비**: 합성 5K~50K 페어 + PII 마스킹 + IAA κ ≥ 0.6 확인.
3. **학습**: Ch 12~15 워크플로우 그대로. 체크포인트 저장.
4. **평가**: perplexity + 도메인 probe + 회귀셋 (Ch 30).
5. **양자화**: int4 GGUF (Ch 19~20). `llama-cli` 로 동작 확인.
6. **Hub 업로드**: 모델 카드 7항목 + GGUF. private → 동작 확인 → public.
7. **(선택) 데모**: HF Spaces 에 Gradio app 배포.
8. **회고**: 한 페이지 작성. 다음 사람이 당신 모델을 Ch 22 결정 트리로 평가할 수 있다.

In [ ]:
# 단계 1~2: 도메인 결정 + 데이터 준비


In [ ]:
# 단계 3~4: 학습 + 평가


In [ ]:
# 단계 5~6: 양자화 + Hub 업로드


In [ ]:
# 단계 7~8: (선택) Spaces 데모 + 회고
